In [1]:
# ============================================================
# Step 6 setup
#
# Upload:
#   1. config.py
#
# Read from Google Drive:
#   2. Step 5 hidden-state zip
#
# This cell prepares:
#   /content/pilot_4/config.py
#   cfg.STEP6_INPUT_DIR
#   cfg.STEP6_OUTPUT_DIR
# ============================================================

import os
import sys
import shutil
import zipfile
import importlib
from pathlib import Path

from google.colab import files
from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# 1. Create project structure
# ------------------------------------------------------------

PILOT_ROOT = "/content/pilot_4"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")
DATA_DIR = os.path.join(PILOT_ROOT, "data")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# ------------------------------------------------------------
# 2. Upload config.py
# ------------------------------------------------------------

print("Upload config.py")
uploaded_config = files.upload()

config_upload_name = None

for name in uploaded_config.keys():
    if name.startswith("config") and name.endswith(".py"):
        config_upload_name = name
        break

if config_upload_name is None:
    raise FileNotFoundError(
        f"No config*.py file was uploaded. Uploaded files: {list(uploaded_config.keys())}"
    )

config_target_path = os.path.join(PILOT_ROOT, "config.py")

shutil.copy(
    os.path.join("/content", config_upload_name),
    config_target_path,
)

# ------------------------------------------------------------
# 3. Load config
# ------------------------------------------------------------

import pilot_4.config as cfg
importlib.reload(cfg)

print("\nLoaded config:")
print("PILOT_ROOT:", cfg.PILOT_ROOT)
print("DATA_DIR:", cfg.DATA_DIR)
print("STEP6_INPUT_DIR:", cfg.STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", cfg.STEP6_OUTPUT_DIR)
print("STEP6_EXPERIMENT_NAME:", cfg.STEP6_EXPERIMENT_NAME)

# ------------------------------------------------------------
# 4. Prepare Step 6 input/output directories
# ------------------------------------------------------------

os.makedirs(cfg.STEP6_INPUT_DIR, exist_ok=True)

# Clear old Step 6 input files.
for old_path in Path(cfg.STEP6_INPUT_DIR).glob("*"):
    if old_path.is_file():
        old_path.unlink()
    elif old_path.is_dir():
        shutil.rmtree(old_path)

# Clear old Step 6 outputs.
if os.path.exists(cfg.STEP6_OUTPUT_DIR):
    shutil.rmtree(cfg.STEP6_OUTPUT_DIR)

os.makedirs(cfg.STEP6_OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 5. Use Step 5 zip from Google Drive
# ------------------------------------------------------------

step5_zip_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expB_settingA1_6fps/data/p4_expB_settingA1_step5_Qwen2_5_3B.zip"
)

print("\nUsing Step 5 zip from Google Drive:")
print(step5_zip_path)

if not step5_zip_path.exists():
    raise FileNotFoundError(f"Step 5 zip not found: {step5_zip_path}")

# ------------------------------------------------------------
# 6. Unzip Step 5 .pt files into STEP6_INPUT_DIR
# ------------------------------------------------------------

with zipfile.ZipFile(str(step5_zip_path), "r") as zf:
    zf.extractall(cfg.STEP6_INPUT_DIR)

pt_files = sorted(Path(cfg.STEP6_INPUT_DIR).rglob("*.pt"))

if not pt_files:
    raise FileNotFoundError(
        f"No .pt files found after extracting Step 5 zip into {cfg.STEP6_INPUT_DIR}"
    )

print("\nSetup complete.")
print("Config copied to:", config_target_path)
print("Step 5 zip:", step5_zip_path)
print("Number of .pt files found:", len(pt_files))
print("STEP6_INPUT_DIR:", cfg.STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", cfg.STEP6_OUTPUT_DIR)

for p in pt_files[:10]:
    print("-", p)

Mounted at /content/drive
Upload config.py


Saving config_p4_expB_settingA1.py to config_p4_expB_settingA1.py

Loaded config:
PILOT_ROOT: /content/pilot_4
DATA_DIR: /content/pilot_4/data
STEP6_INPUT_DIR: /content/pilot_4/data/step6_expB_settingA1_input_hidden_states
STEP6_OUTPUT_DIR: /content/pilot_4/data/step6_expB_settingA1_implicit_transitive_outputs
STEP6_EXPERIMENT_NAME: expB_settingA1_implicit_transitive_scene_split_lr_probe

Using Step 5 zip from Google Drive:
/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expB_settingA1_6fps/data/p4_expB_settingA1_step5_Qwen2_5_3B.zip

Setup complete.
Config copied to: /content/pilot_4/config.py
Step 5 zip: /content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expB_settingA1_6fps/data/p4_expB_settingA1_step5_Qwen2_5_3B.zip
Number of .pt files found: 6
STEP6_INPUT_DIR: /content/pilot_4/data/step6_expB_settingA1_input_hidden_states
STEP6_OUTPUT_DIR: /content/pilot_4/data/step6_expB_settingA1_implicit_transitive_outputs
- /content/pilot_4/data/step6_expB_settingA1_input_hid

In [2]:
# ============================================================
# Imports and Step 6 configuration
#
# Current Step 6 goal:
#   Decode implicit-transitive spatial relation from Step5 hidden states.
#
# Main task:
#   X = hidden(probe_subject) - hidden(probe_object)
#   y = relation in {left_of, right_of}
# ============================================================

import os
import sys
import json
import random
import importlib
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_4.config as cfg
importlib.reload(cfg)

# ------------------------------------------------------------
# Basic config
# ------------------------------------------------------------

EXPERIMENT_NAME = getattr(
    cfg,
    "STEP6_EXPERIMENT_NAME",
    "expB_settingA1_implicit_transitive_scene_split_lr_probe",
)

RANDOM_SEED = getattr(cfg, "RANDOM_SEED", 42)

MODEL_NAME = getattr(cfg, "STEP5_MODEL_NAME", "Qwen/Qwen2.5-3B")
MODEL_TAG = getattr(cfg, "STEP5_MODEL_TAG", "Qwen2_5_3B")

STEP6_INPUT_DIR = cfg.STEP6_INPUT_DIR
STEP6_OUTPUT_DIR = cfg.STEP6_OUTPUT_DIR

FEATURE_KEY = getattr(cfg, "STEP6_FEATURE_KEY", "layer_diff_features")
LABEL_FIELD = getattr(cfg, "STEP6_LABEL_FIELD", "relation")

TRAIN_SCENES = list(getattr(
    cfg,
    "STEP6_TRAIN_SCENES",
    ["FloorPlan1", "FloorPlan2", "FloorPlan3", "FloorPlan4"],
))

TEST_SCENES = list(getattr(
    cfg,
    "STEP6_TEST_SCENES",
    ["FloorPlan5", "FloorPlan6"],
))

REQUIRE_EXPLICIT_SCENE_SPLIT = getattr(
    cfg,
    "STEP6_REQUIRE_EXPLICIT_SCENE_SPLIT",
    True,
)

LOGREG_MAX_ITER = getattr(cfg, "STEP6_LOGREG_MAX_ITER", 5000)
LOGREG_C = getattr(cfg, "STEP6_LOGREG_C", 1.0)

PRINT_DATASET_SUMMARY = getattr(cfg, "STEP6_PRINT_DATASET_SUMMARY", True)
PRINT_LAYER_PROGRESS = getattr(cfg, "STEP6_PRINT_LAYER_PROGRESS", True)
PRINT_TOP_LAYERS = getattr(cfg, "STEP6_PRINT_TOP_LAYERS", True)
NUM_TOP_LAYERS_TO_PRINT = getattr(cfg, "STEP6_NUM_TOP_LAYERS_TO_PRINT", 10)

# ------------------------------------------------------------
# Current ExpB SettingA1 filtering
# ------------------------------------------------------------

IMPLICIT_TRANSITIVE = getattr(cfg, "IMPLICIT_TRANSITIVE", "implicit_transitive")

ALLOWED_EVIDENCE_TYPES = set(getattr(
    cfg,
    "STEP6_ALLOWED_EVIDENCE_TYPES",
    {IMPLICIT_TRANSITIVE},
))

ALLOWED_LABELS = set(getattr(
    cfg,
    "STEP6_ALLOWED_LABELS",
    {"left_of", "right_of"},
))

# Optional: group test results by implicit reasoning rule.
GROUP_BY_IMPLICIT_RULE = getattr(
    cfg,
    "STEP6_GROUP_BY_IMPLICIT_RULE",
    True,
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.makedirs(STEP6_OUTPUT_DIR, exist_ok=True)

print("Step 6 config loaded.")
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_TAG:", MODEL_TAG)
print("STEP6_INPUT_DIR:", STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", STEP6_OUTPUT_DIR)
print("FEATURE_KEY:", FEATURE_KEY)
print("LABEL_FIELD:", LABEL_FIELD)
print("TRAIN_SCENES:", TRAIN_SCENES)
print("TEST_SCENES:", TEST_SCENES)

print("\nCurrent Step6 task:")
print("Decode implicit-transitive relation from hidden-state features.")
print("ALLOWED_EVIDENCE_TYPES:", sorted(ALLOWED_EVIDENCE_TYPES))
print("ALLOWED_LABELS:", sorted(ALLOWED_LABELS))
print("GROUP_BY_IMPLICIT_RULE:", GROUP_BY_IMPLICIT_RULE)

Step 6 config loaded.
EXPERIMENT_NAME: expB_settingA1_implicit_transitive_scene_split_lr_probe
MODEL_NAME: Qwen/Qwen2.5-3B
MODEL_TAG: Qwen2_5_3B
STEP6_INPUT_DIR: /content/pilot_4/data/step6_expB_settingA1_input_hidden_states
STEP6_OUTPUT_DIR: /content/pilot_4/data/step6_expB_settingA1_implicit_transitive_outputs
FEATURE_KEY: layer_diff_features
LABEL_FIELD: relation
TRAIN_SCENES: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4']
TEST_SCENES: ['FloorPlan5', 'FloorPlan6']

Current Step6 task:
Decode implicit-transitive relation from hidden-state features.
ALLOWED_EVIDENCE_TYPES: ['implicit_transitive']
ALLOWED_LABELS: ['left_of', 'right_of']
GROUP_BY_IMPLICIT_RULE: True


In [3]:
# ============================================================
# Helper functions for loading Step 5 records
# ============================================================

def discover_step5_pt_files(input_dir: str) -> List[str]:
    pt_paths = sorted(Path(input_dir).rglob("*.pt"))

    if not pt_paths:
        raise FileNotFoundError(f"No .pt files found in STEP6_INPUT_DIR: {input_dir}")

    return [str(p) for p in pt_paths]


def normalize_record(raw_record: Dict[str, Any], source_pt_file: str) -> Dict[str, Any]:
    """
    Normalize one Step 5 record into a predictable structure.

    Expected current Step5 fields:
      - layer_diff_features / layer_concat_features / ...
      - relation
      - scene
      - evidence_type = implicit_transitive
      - implicit_rule
      - anchor_uid
      - subject_alias / object_alias
    """

    rec = dict(raw_record)
    rec["_source_pt_file"] = source_pt_file

    # Some versions save metadata as rec["metadata"], while others flatten it.
    metadata = rec.get("metadata", {})

    if metadata is None:
        metadata = {}

    if not isinstance(metadata, dict):
        metadata = {}

    for k, v in metadata.items():
        rec.setdefault(k, v)

    # If nested evidence exists, expose useful fields.
    evidence = rec.get("evidence", {})
    if isinstance(evidence, dict):
        rec.setdefault("evidence_type", evidence.get("evidence_type"))
        rec.setdefault(
            "probe_direction_relative_to_text",
            evidence.get("probe_direction_relative_to_text"),
        )
        rec.setdefault("implicit_rule", evidence.get("implicit_rule"))
        rec.setdefault("anchor_uid", evidence.get("anchor_uid"))
        rec.setdefault("supporting_relations", evidence.get("supporting_relations"))
        rec.setdefault("num_supporting_paths", evidence.get("num_supporting_paths"))

    # If nested probe_pair exists, expose useful fields.
    probe_pair = rec.get("probe_pair", {})
    if isinstance(probe_pair, dict):
        rec.setdefault("probe_subject_uid", probe_pair.get("probe_subject_uid"))
        rec.setdefault("probe_object_uid", probe_pair.get("probe_object_uid"))
        rec.setdefault("probe_relation_label", probe_pair.get("probe_relation_label"))
        rec.setdefault("is_implicit_transitive", probe_pair.get("is_implicit_transitive"))
        rec.setdefault("is_explicit_in_text", probe_pair.get("is_explicit_in_text"))

    # Standardize relation label.
    if LABEL_FIELD not in rec:
        if "relation" in rec:
            rec[LABEL_FIELD] = rec["relation"]
        elif "probe_relation_label" in rec:
            rec[LABEL_FIELD] = rec["probe_relation_label"]
        elif isinstance(probe_pair, dict) and "probe_relation_label" in probe_pair:
            rec[LABEL_FIELD] = probe_pair["probe_relation_label"]
        else:
            raise KeyError(f"Could not find label field '{LABEL_FIELD}' in record.")

    # Standardize scene.
    if "scene" not in rec:
        if "source_step3_scene" in rec:
            rec["scene"] = rec["source_step3_scene"]
        else:
            raise KeyError("Could not find scene field in record.")

    # Standardize subject/object aliases if needed.
    if "subject_alias" not in rec:
        rec["subject_alias"] = (
            rec.get("probe_subject_uid")
            or rec.get("subject_uid")
            or rec.get("subject")
        )

    if "object_alias" not in rec:
        rec["object_alias"] = (
            rec.get("probe_object_uid")
            or rec.get("object_uid")
            or rec.get("object")
        )

    return rec


def extract_records_from_payload(payload: Any, source_pt_file: str) -> List[Dict[str, Any]]:
    """
    Supports several possible Step 5 payload formats:
      - list[record]
      - dict with key "records"
      - dict with key "data"
      - dict with key "examples"
    """

    if isinstance(payload, list):
        raw_records = payload
    elif isinstance(payload, dict):
        if "records" in payload:
            raw_records = payload["records"]
        elif "data" in payload:
            raw_records = payload["data"]
        elif "examples" in payload:
            raw_records = payload["examples"]
        else:
            raise KeyError(
                f"Unsupported .pt payload keys in {source_pt_file}: {list(payload.keys())}"
            )
    else:
        raise TypeError(f"Unsupported .pt payload type in {source_pt_file}: {type(payload)}")

    records = [
        normalize_record(r, source_pt_file=source_pt_file)
        for r in raw_records
    ]

    return records


def to_numpy_feature(x: Any) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().float().numpy()

    arr = np.asarray(x)

    if arr.dtype == np.dtype("O"):
        arr = arr.astype(np.float32)

    return arr.astype(np.float32, copy=False)


print("Helper functions loaded.")

Helper functions loaded.


In [4]:
# ============================================================
# Load Step 5 records, filter implicit-transitive samples,
# and build feature tensor
# ============================================================

pt_files = discover_step5_pt_files(STEP6_INPUT_DIR)

all_records_raw = []

for pt_path in pt_files:
    payload = torch.load(pt_path, map_location="cpu")
    records = extract_records_from_payload(payload, source_pt_file=pt_path)
    all_records_raw.extend(records)

if not all_records_raw:
    raise ValueError("No Step 5 records loaded.")

print("Number of raw Step 5 records loaded:", len(all_records_raw))
print("Number of source .pt files:", len(pt_files))

# ------------------------------------------------------------
# Filter to current ExpB SettingA1 target samples
# ------------------------------------------------------------

filtered_records = []

for r in all_records_raw:
    evidence_type = r.get("evidence_type")
    label = r.get(LABEL_FIELD)

    if evidence_type not in ALLOWED_EVIDENCE_TYPES:
        continue

    if label not in ALLOWED_LABELS:
        continue

    filtered_records.append(r)

if not filtered_records:
    raise ValueError(
        "No records left after filtering. "
        f"ALLOWED_EVIDENCE_TYPES={ALLOWED_EVIDENCE_TYPES}, "
        f"ALLOWED_LABELS={ALLOWED_LABELS}"
    )

all_records = filtered_records

print("\nNumber of records after filtering:", len(all_records))

# ------------------------------------------------------------
# Validate feature and label fields
# ------------------------------------------------------------

missing_feature = [i for i, r in enumerate(all_records) if FEATURE_KEY not in r]
missing_label = [i for i, r in enumerate(all_records) if LABEL_FIELD not in r]

if missing_feature:
    raise KeyError(f"{len(missing_feature)} records are missing feature key: {FEATURE_KEY}")

if missing_label:
    raise KeyError(f"{len(missing_label)} records are missing label field: {LABEL_FIELD}")

# ------------------------------------------------------------
# Stack features
# Expected shape per record: [num_layers, hidden_dim]
# ------------------------------------------------------------

features = []

for r in all_records:
    f = to_numpy_feature(r[FEATURE_KEY])

    if f.ndim != 2:
        raise ValueError(
            f"Expected feature shape [num_layers, dim], got {f.shape} "
            f"for sample_id={r.get('sample_id')}"
        )

    features.append(f)

X_all = np.stack(features, axis=0)  # [num_samples, num_layers, dim]
y_all = np.array([r[LABEL_FIELD] for r in all_records])
metadata = all_records

num_samples, num_layers, feature_dim = X_all.shape

print("\nX_all shape:", X_all.shape)
print("num_samples:", num_samples)
print("num_layers:", num_layers)
print("feature_dim:", feature_dim)

print("\nLabel counts:")
print(dict(Counter(y_all)))

print("\nScene counts:")
print(dict(Counter(r["scene"] for r in metadata)))

print("\nEvidence type counts:")
print(dict(Counter(r.get("evidence_type", "unknown") for r in metadata)))

print("\nImplicit rule counts:")
print(dict(Counter(r.get("implicit_rule", "unknown") for r in metadata)))

print("\nProbe direction counts:")
print(dict(Counter(r.get("probe_direction_relative_to_text", "unknown") for r in metadata)))

Number of raw Step 5 records loaded: 167
Number of source .pt files: 6

Number of records after filtering: 167

X_all shape: (167, 37, 2048)
num_samples: 167
num_layers: 37
feature_dim: 2048

Label counts:
{np.str_('left_of'): 86, np.str_('right_of'): 81}

Scene counts:
{'FloorPlan1': 30, 'FloorPlan2': 23, 'FloorPlan3': 30, 'FloorPlan4': 40, 'FloorPlan5': 31, 'FloorPlan6': 13}

Evidence type counts:
{'implicit_transitive': 167}

Implicit rule counts:
{'chain_same_direction': 83, 'shared_anchor_opposite_sides': 37, 'anchor_between_reversed_surface_form': 47}

Probe direction counts:
{'implicit': 167}


In [5]:
# ============================================================
# Scene split and common-label filtering
#
# Goal:
#   train on TRAIN_SCENES
#   test on TEST_SCENES
#
# Labels:
#   relation in {left_of, right_of}
# ============================================================

all_scenes = sorted(set(r["scene"] for r in metadata))

if REQUIRE_EXPLICIT_SCENE_SPLIT:
    missing_train = sorted(set(TRAIN_SCENES) - set(all_scenes))
    missing_test = sorted(set(TEST_SCENES) - set(all_scenes))

    if missing_train or missing_test:
        raise ValueError(
            f"Scene split references missing scenes. "
            f"missing_train={missing_train}, missing_test={missing_test}, "
            f"available_scenes={all_scenes}"
        )

train_idx = np.array(
    [i for i, r in enumerate(metadata) if r["scene"] in TRAIN_SCENES],
    dtype=np.int64,
)

test_idx = np.array(
    [i for i, r in enumerate(metadata) if r["scene"] in TEST_SCENES],
    dtype=np.int64,
)

if len(train_idx) == 0:
    raise ValueError("No training examples selected.")

if len(test_idx) == 0:
    raise ValueError("No test examples selected.")

train_labels_before = set(y_all[train_idx])
test_labels_before = set(y_all[test_idx])
common_labels = sorted(train_labels_before & test_labels_before)

if len(common_labels) < 2:
    raise ValueError(
        f"Need at least two common labels for classification. "
        f"train_labels={sorted(train_labels_before)}, "
        f"test_labels={sorted(test_labels_before)}, "
        f"common_labels={common_labels}"
    )

removed_train_only_labels = sorted(train_labels_before - test_labels_before)
removed_test_only_labels = sorted(test_labels_before - train_labels_before)

train_idx = np.array(
    [idx for idx in train_idx if y_all[idx] in common_labels],
    dtype=np.int64,
)

test_idx = np.array(
    [idx for idx in test_idx if y_all[idx] in common_labels],
    dtype=np.int64,
)

label_order = common_labels

scene_split_info = {
    "train_scenes": TRAIN_SCENES,
    "test_scenes": TEST_SCENES,
    "available_scenes": all_scenes,

    "num_train_before_common_label_filter": int(sum(r["scene"] in TRAIN_SCENES for r in metadata)),
    "num_test_before_common_label_filter": int(sum(r["scene"] in TEST_SCENES for r in metadata)),

    "train_labels_before": sorted(train_labels_before),
    "test_labels_before": sorted(test_labels_before),
    "common_labels": common_labels,
    "removed_train_only_labels": removed_train_only_labels,
    "removed_test_only_labels": removed_test_only_labels,

    "num_train_final": int(len(train_idx)),
    "num_test_final": int(len(test_idx)),

    "train_label_counts_final": dict(Counter(y_all[train_idx])),
    "test_label_counts_final": dict(Counter(y_all[test_idx])),

    "train_scene_counts_final": dict(Counter(metadata[int(idx)]["scene"] for idx in train_idx)),
    "test_scene_counts_final": dict(Counter(metadata[int(idx)]["scene"] for idx in test_idx)),

    "train_evidence_type_counts_final": dict(
        Counter(metadata[int(idx)].get("evidence_type", "unknown") for idx in train_idx)
    ),
    "test_evidence_type_counts_final": dict(
        Counter(metadata[int(idx)].get("evidence_type", "unknown") for idx in test_idx)
    ),

    "train_implicit_rule_counts_final": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in train_idx)
    ),
    "test_implicit_rule_counts_final": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in test_idx)
    ),
}

print("Scene split complete.")
print(json.dumps(scene_split_info, indent=2, ensure_ascii=False))

print("\nFinal label_order:")
print(label_order)

print("\nFinal train label counts:")
print(dict(Counter(y_all[train_idx])))

print("\nFinal test label counts:")
print(dict(Counter(y_all[test_idx])))

print("\nFinal train implicit rule counts:")
print(dict(Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in train_idx)))

print("\nFinal test implicit rule counts:")
print(dict(Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in test_idx)))

Scene split complete.
{
  "train_scenes": [
    "FloorPlan1",
    "FloorPlan2",
    "FloorPlan3",
    "FloorPlan4"
  ],
  "test_scenes": [
    "FloorPlan5",
    "FloorPlan6"
  ],
  "available_scenes": [
    "FloorPlan1",
    "FloorPlan2",
    "FloorPlan3",
    "FloorPlan4",
    "FloorPlan5",
    "FloorPlan6"
  ],
  "num_train_before_common_label_filter": 123,
  "num_test_before_common_label_filter": 44,
  "train_labels_before": [
    "left_of",
    "right_of"
  ],
  "test_labels_before": [
    "left_of",
    "right_of"
  ],
  "common_labels": [
    "left_of",
    "right_of"
  ],
  "removed_train_only_labels": [],
  "removed_test_only_labels": [],
  "num_train_final": 123,
  "num_test_final": 44,
  "train_label_counts_final": {
    "left_of": 68,
    "right_of": 55
  },
  "test_label_counts_final": {
    "left_of": 18,
    "right_of": 26
  },
  "train_scene_counts_final": {
    "FloorPlan1": 30,
    "FloorPlan2": 23,
    "FloorPlan3": 30,
    "FloorPlan4": 40
  },
  "test_scene_counts_f

In [6]:
# ============================================================
# No direct/inverse direction filtering for current Step6
#
# Current data:
#   evidence_type = implicit_transitive
#   probe_direction_relative_to_text = implicit
#
# Therefore, old direct/inverse filtering is disabled.
# ============================================================

direction_filter_info = {
    "enabled": False,
    "reason": (
        "Current Step6 uses Step4T implicit-transitive samples. "
        "There is no direct/inverse train-test direction protocol here."
    ),
}

print("Direction filtering disabled for current implicit-transitive Step6.")
print(json.dumps(direction_filter_info, indent=2, ensure_ascii=False))

Direction filtering disabled for current implicit-transitive Step6.
{
  "enabled": false,
  "reason": "Current Step6 uses Step4T implicit-transitive samples. There is no direct/inverse train-test direction protocol here."
}


In [7]:
# ============================================================
# Final train/test split summary
# ============================================================

final_split_summary = {
    "num_samples_loaded_after_filtering": int(num_samples),
    "num_train": int(len(train_idx)),
    "num_test": int(len(test_idx)),
    "label_order": label_order,

    "train_label_counts": dict(Counter(y_all[train_idx])),
    "test_label_counts": dict(Counter(y_all[test_idx])),

    "train_scene_counts": dict(Counter(metadata[int(idx)]["scene"] for idx in train_idx)),
    "test_scene_counts": dict(Counter(metadata[int(idx)]["scene"] for idx in test_idx)),

    "train_implicit_rule_counts": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in train_idx)
    ),
    "test_implicit_rule_counts": dict(
        Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in test_idx)
    ),

    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,
}

print("Final train/test split summary:")
print(json.dumps(final_split_summary, indent=2, ensure_ascii=False))

Final train/test split summary:
{
  "num_samples_loaded_after_filtering": 167,
  "num_train": 123,
  "num_test": 44,
  "label_order": [
    "left_of",
    "right_of"
  ],
  "train_label_counts": {
    "left_of": 68,
    "right_of": 55
  },
  "test_label_counts": {
    "left_of": 18,
    "right_of": 26
  },
  "train_scene_counts": {
    "FloorPlan1": 30,
    "FloorPlan2": 23,
    "FloorPlan3": 30,
    "FloorPlan4": 40
  },
  "test_scene_counts": {
    "FloorPlan5": 31,
    "FloorPlan6": 13
  },
  "train_implicit_rule_counts": {
    "chain_same_direction": 60,
    "shared_anchor_opposite_sides": 28,
    "anchor_between_reversed_surface_form": 35
  },
  "test_implicit_rule_counts": {
    "anchor_between_reversed_surface_form": 12,
    "chain_same_direction": 23,
    "shared_anchor_opposite_sides": 9
  },
  "feature_key": "layer_diff_features",
  "label_field": "relation"
}


In [8]:
# ============================================================
# Build test subsets
#
# Current subsets:
#   - overall test set
#   - optional subsets by implicit_rule
# ============================================================

test_subset_indices = {
    "test_overall": test_idx,
}

if GROUP_BY_IMPLICIT_RULE:
    test_rules = sorted(
        set(
            metadata[int(idx)].get("implicit_rule", "unknown")
            for idx in test_idx
        )
    )

    for rule in test_rules:
        subset_key = f"test_rule_{rule}"
        rule_idx = np.array(
            [
                idx for idx in test_idx
                if metadata[int(idx)].get("implicit_rule", "unknown") == rule
            ],
            dtype=np.int64,
        )
        test_subset_indices[subset_key] = rule_idx

test_subset_summary = {}

for subset_key, idxs in test_subset_indices.items():
    test_subset_summary[subset_key] = {
        "num_examples": int(len(idxs)),
        "label_counts": dict(Counter(y_all[idxs])) if len(idxs) else {},
        "evidence_type_counts": dict(
            Counter(metadata[int(idx)].get("evidence_type", "unknown") for idx in idxs)
        ) if len(idxs) else {},
        "implicit_rule_counts": dict(
            Counter(metadata[int(idx)].get("implicit_rule", "unknown") for idx in idxs)
        ) if len(idxs) else {},
        "scene_counts": dict(
            Counter(metadata[int(idx)].get("scene", "unknown") for idx in idxs)
        ) if len(idxs) else {},
    }

print("Test subset summary:")
print(json.dumps(test_subset_summary, indent=2, ensure_ascii=False))

Test subset summary:
{
  "test_overall": {
    "num_examples": 44,
    "label_counts": {
      "left_of": 18,
      "right_of": 26
    },
    "evidence_type_counts": {
      "implicit_transitive": 44
    },
    "implicit_rule_counts": {
      "anchor_between_reversed_surface_form": 12,
      "chain_same_direction": 23,
      "shared_anchor_opposite_sides": 9
    },
    "scene_counts": {
      "FloorPlan5": 31,
      "FloorPlan6": 13
    }
  },
  "test_rule_anchor_between_reversed_surface_form": {
    "num_examples": 12,
    "label_counts": {
      "left_of": 6,
      "right_of": 6
    },
    "evidence_type_counts": {
      "implicit_transitive": 12
    },
    "implicit_rule_counts": {
      "anchor_between_reversed_surface_form": 12
    },
    "scene_counts": {
      "FloorPlan5": 10,
      "FloorPlan6": 2
    }
  },
  "test_rule_chain_same_direction": {
    "num_examples": 23,
    "label_counts": {
      "right_of": 16,
      "left_of": 7
    },
    "evidence_type_counts": {
      "im

In [9]:
# ============================================================
# Layer-wise training and evaluation
# ============================================================

def evaluate_classifier(
    clf: LogisticRegression,
    X_eval: np.ndarray,
    y_eval: np.ndarray,
    labels: List[str],
) -> Dict[str, Any]:
    if len(y_eval) == 0:
        return {
            "num_examples": 0,
            "accuracy": None,
            "macro_f1": None,
            "label_order": labels,
            "classification_report": {},
            "confusion_matrix": [],
        }

    y_pred = clf.predict(X_eval)

    acc = accuracy_score(y_eval, y_pred)
    macro_f1 = f1_score(
        y_eval,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )

    report = classification_report(
        y_eval,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_eval,
        y_pred,
        labels=labels,
    )

    return {
        "num_examples": int(len(y_eval)),
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "label_order": labels,
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
    }


results_by_layer = []

for layer_id in range(num_layers):
    if PRINT_LAYER_PROGRESS:
        print(f"Training layer {layer_id}/{num_layers - 1}")

    X_train_layer = X_all[train_idx, layer_id, :]
    y_train_layer = y_all[train_idx]

    clf = LogisticRegression(
        max_iter=LOGREG_MAX_ITER,
        C=LOGREG_C,
        multi_class="auto",
        solver="lbfgs",
        random_state=RANDOM_SEED,
    )

    clf.fit(X_train_layer, y_train_layer)

    layer_result = {
        "layer": int(layer_id),
        "train": evaluate_classifier(
            clf,
            X_train_layer,
            y_train_layer,
            labels=label_order,
        ),
    }

    for subset_key, idxs in test_subset_indices.items():
        X_eval = X_all[idxs, layer_id, :]
        y_eval = y_all[idxs]

        layer_result[subset_key] = evaluate_classifier(
            clf,
            X_eval,
            y_eval,
            labels=label_order,
        )

    results_by_layer.append(layer_result)

print("Layer-wise training and evaluation complete.")

Training layer 0/36
Training layer 1/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 2/36
Training layer 3/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 4/36
Training layer 5/36


...

Training layer 33/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 34/36
Training layer 35/36
Training layer 36/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer-wise training and evaluation complete.


In [10]:
# ============================================================
# Summarize top layers
# ============================================================

summary_rows = []

test_subset_keys = list(test_subset_indices.keys())

for r in results_by_layer:
    row = {
        "layer": r["layer"],
        "train_accuracy": r["train"]["accuracy"],
        "train_macro_f1": r["train"]["macro_f1"],
    }

    for subset_key in test_subset_keys:
        row[f"{subset_key}_accuracy"] = r[subset_key]["accuracy"]
        row[f"{subset_key}_macro_f1"] = r[subset_key]["macro_f1"]
        row[f"{subset_key}_num_examples"] = r[subset_key]["num_examples"]

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_csv_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}_layer_summary.csv",
)

summary_df.to_csv(summary_csv_path, index=False)

print("Saved layer summary CSV:")
print(summary_csv_path)

display(summary_df.head())

if PRINT_TOP_LAYERS:
    ranking_metrics = [
        "test_overall_accuracy",
        "test_overall_macro_f1",
    ]

    # Also print rule-specific metrics if available.
    for subset_key in test_subset_keys:
        if subset_key != "test_overall":
            ranking_metrics.append(f"{subset_key}_accuracy")

    for metric in ranking_metrics:
        if metric not in summary_df.columns:
            continue

        print("\n" + "=" * 80)
        print("Top layers by:", metric)
        print("=" * 80)

        cols = [
            "layer",
            metric,
            "test_overall_accuracy",
            "test_overall_macro_f1",
        ]

        # Add available subset accuracy columns.
        for subset_key in test_subset_keys:
            acc_col = f"{subset_key}_accuracy"
            if acc_col not in cols and acc_col in summary_df.columns:
                cols.append(acc_col)

        display(
            summary_df.sort_values(metric, ascending=False)
            .loc[:, cols]
            .head(NUM_TOP_LAYERS_TO_PRINT)
        )

Saved layer summary CSV:
/content/pilot_4/data/step6_expB_settingA1_implicit_transitive_outputs/step6_expB_settingA1_implicit_transitive_scene_split_lr_probe_Qwen2_5_3B_layer_diff_features_layer_summary.csv


,layer,train_accuracy,train_macro_f1,test_overall_accuracy,test_overall_macro_f1,test_overall_num_examples,test_rule_anchor_between_reversed_surface_form_accuracy,test_rule_anchor_between_reversed_surface_form_macro_f1,test_rule_anchor_between_reversed_surface_form_num_examples,test_rule_chain_same_direction_accuracy,test_rule_chain_same_direction_macro_f1,test_rule_chain_same_direction_num_examples,test_rule_shared_anchor_opposite_sides_accuracy,test_rule_shared_anchor_opposite_sides_macro_f1,test_rule_shared_anchor_opposite_sides_num_examples
0,0,0.886179,0.884461,0.409091,0.388889,44,0.500000,0.437500,12,0.304348,0.303030,23,0.555556,0.357143,9
1,1,1.000000,1.000000,0.545455,0.541667,44,0.666667,0.666667,12,0.434783,0.417154,23,0.666667,0.666667,9
2,2,1.000000,1.000000,0.500000,0.495833,44,0.666667,0.666667,12,0.434783,0.417154,23,0.444444,0.444444,9
3,3,1.000000,1.000000,0.500000,0.498965,44,0.666667,0.666667,12,0.391304,0.380769,23,0.555556,0.550000,9
4,4,1.000000,1.000000,0.454545,0.453416,44,0.666667,0.666667,12,0.347826,0.342857,23,0.444444,0.444444,9



Top layers by: test_overall_accuracy


,layer,test_overall_accuracy,test_overall_accuracy,test_overall_macro_f1,test_rule_anchor_between_reversed_surface_form_accuracy,test_rule_chain_same_direction_accuracy,test_rule_shared_anchor_opposite_sides_accuracy
1,1,0.545455,0.545455,0.541667,0.666667,0.434783,0.666667
5,5,0.522727,0.522727,0.520498,0.666667,0.391304,0.666667
3,3,0.500000,0.500000,0.498965,0.666667,0.391304,0.555556
19,19,0.500000,0.500000,0.498965,0.500000,0.347826,0.888889
2,2,0.500000,0.500000,0.495833,0.666667,0.434783,0.444444
6,6,0.454545,0.454545,0.453416,0.500000,0.347826,0.666667
4,4,0.454545,0.454545,0.453416,0.666667,0.347826,0.444444
36,36,0.431818,0.431818,0.431525,0.333333,0.391304,0.666667
7,7,0.431818,0.431818,0.431525,0.500000,0.304348,0.666667
17,17,0.431818,0.431818,0.429165,0.666667,0.260870,0.555556



Top layers by: test_overall_macro_f1


...

,layer,test_rule_anchor_between_reversed_surface_form_accuracy,test_overall_accuracy,test_overall_macro_f1,test_rule_chain_same_direction_accuracy,test_rule_shared_anchor_opposite_sides_accuracy
1,1,0.666667,0.545455,0.541667,0.434783,0.666667
3,3,0.666667,0.500000,0.498965,0.391304,0.555556
2,2,0.666667,0.500000,0.495833,0.434783,0.444444
4,4,0.666667,0.454545,0.453416,0.347826,0.444444
5,5,0.666667,0.522727,0.520498,0.391304,0.666667
17,17,0.666667,0.431818,0.429165,0.260870,0.555556
0,0,0.500000,0.409091,0.388889,0.304348,0.555556
7,7,0.500000,0.431818,0.431525,0.304348,0.666667
6,6,0.500000,0.454545,0.453416,0.347826,0.666667
19,19,0.500000,0.500000,0.498965,0.347826,0.888889



Top layers by: test_rule_chain_same_direction_accuracy


,layer,test_rule_chain_same_direction_accuracy,test_overall_accuracy,test_overall_macro_f1,test_rule_anchor_between_reversed_surface_form_accuracy,test_rule_shared_anchor_opposite_sides_accuracy
1,1,0.434783,0.545455,0.541667,0.666667,0.666667
2,2,0.434783,0.500000,0.495833,0.666667,0.444444
3,3,0.391304,0.500000,0.498965,0.666667,0.555556
11,11,0.391304,0.363636,0.363636,0.250000,0.444444
5,5,0.391304,0.522727,0.520498,0.666667,0.666667
36,36,0.391304,0.431818,0.431525,0.333333,0.666667
35,35,0.391304,0.431818,0.431525,0.333333,0.666667
34,34,0.391304,0.431818,0.431525,0.333333,0.666667
32,32,0.391304,0.431818,0.431525,0.333333,0.666667
33,33,0.391304,0.431818,0.431525,0.333333,0.666667



Top layers by: test_rule_shared_anchor_opposite_sides_accuracy


,layer,test_rule_shared_anchor_opposite_sides_accuracy,test_overall_accuracy,test_overall_macro_f1,test_rule_anchor_between_reversed_surface_form_accuracy,test_rule_chain_same_direction_accuracy
19,19,0.888889,0.500000,0.498965,0.500000,0.347826
20,20,0.777778,0.409091,0.409091,0.333333,0.304348
5,5,0.666667,0.522727,0.520498,0.666667,0.391304
6,6,0.666667,0.454545,0.453416,0.500000,0.347826
1,1,0.666667,0.545455,0.541667,0.666667,0.434783
35,35,0.666667,0.431818,0.431525,0.333333,0.391304
36,36,0.666667,0.431818,0.431525,0.333333,0.391304
31,31,0.666667,0.386364,0.386047,0.416667,0.260870
32,32,0.666667,0.431818,0.431525,0.333333,0.391304
9,9,0.666667,0.409091,0.407867,0.333333,0.347826


In [11]:
# ============================================================
# Save full Step 6 result JSON
# ============================================================

output_payload = {
    "experiment_name": EXPERIMENT_NAME,
    "description": (
        "ExpB SettingA1 implicit-transitive relation decoding. "
        "A linear probe is trained on Step5 hidden-state features to predict "
        "the implicit spatial relation label between probe_subject and probe_object. "
        "The main feature is hidden(probe_subject) - hidden(probe_object). "
        "Evaluation uses a scene split and optionally reports results by implicit_rule."
    ),

    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,

    "probe_target": {
        "task": "implicit_transitive_relation_decoding",
        "input_feature": FEATURE_KEY,
        "label": LABEL_FIELD,
        "allowed_labels": sorted(ALLOWED_LABELS),
        "allowed_evidence_types": sorted(ALLOWED_EVIDENCE_TYPES),
        "feature_interpretation": "hidden(probe_subject) - hidden(probe_object)",
    },

    "num_samples_loaded_after_filtering": int(num_samples),
    "num_layers": int(num_layers),
    "feature_dim": int(feature_dim),
    "label_order": label_order,

    "scene_split_info": scene_split_info,
    "direction_filter_info": direction_filter_info,
    "final_split_summary": final_split_summary,
    "test_subset_summary": test_subset_summary,

    "step6_config": {
        "experiment_name": EXPERIMENT_NAME,
        "model_name": MODEL_NAME,
        "model_tag": MODEL_TAG,
        "feature_key": FEATURE_KEY,
        "label_field": LABEL_FIELD,
        "train_scenes": TRAIN_SCENES,
        "test_scenes": TEST_SCENES,
        "logreg_max_iter": LOGREG_MAX_ITER,
        "logreg_c": LOGREG_C,
        "random_seed": RANDOM_SEED,
        "group_by_implicit_rule": GROUP_BY_IMPLICIT_RULE,
    },

    "source_pt_files": pt_files,
    "results_by_layer": results_by_layer,
}

output_json_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}.json",
)

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, indent=2, ensure_ascii=False)

print("Saved full Step 6 result JSON:")
print(output_json_path)

Saved full Step 6 result JSON:
/content/pilot_4/data/step6_expB_settingA1_implicit_transitive_outputs/step6_expB_settingA1_implicit_transitive_scene_split_lr_probe_Qwen2_5_3B_layer_diff_features.json


In [12]:
# ============================================================
# Zip Step 6 outputs
# ============================================================

zip_base = f"/content/p4_expB_settingA1_step6_lr_{MODEL_TAG}_{FEATURE_KEY}"

zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=STEP6_OUTPUT_DIR,
)

print("Created Step 6 output zip:")
print(zip_path)

Created Step 6 output zip:
/content/p4_expB_settingA1_step6_lr_Qwen2_5_3B_layer_diff_features.zip


In [13]:
# ============================================================
# Optional download
# ============================================================

from google.colab import files

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>